<a href="https://colab.research.google.com/github/lim0119/-2025-3-2-PJ/blob/main/new_%ED%92%88%EC%A7%88_%EC%BD%94%EB%93%9C(%EB%8B%A8%EC%9C%84_%ED%85%8C%EC%8A%A4%ED%8A%B8).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile test_quality_unit.py
import pytest
import numpy as np
import time
import random
from qa_metrics import compute_metrics # 성능 지표 계산 함수

def calculate_icv(mri_data: np.ndarray) -> float:
    """ICV(두개강 용적) 실시간 계산 로직 모방"""
    # 실제 서비스에서는 MRI 전체 볼륨을 기반으로 ICV를 계산하는 공식이 존재함.
    # 여기서는 단위 테스트용으로 랜덤 스케일을 곱한 더미 계산 로직을 사용하여
    # "ICV 값이 정상적으로 계산되는지"만 확인하기 위한 시뮬레이션.
    return mri_data.sum() * 0.001 * random.uniform(500, 1000)  # 더미 계산


def segment_hippocampus_performance(data_array: np.ndarray, processing_time_sec: float = 14.0) -> np.ndarray:
    """해마 세그멘테이션 성능 시뮬레이션 (목표 15초 이내)"""
    # 실제 해마 세그멘테이션 알고리즘의 처리 시간을 단위 테스트에서 대체하기 위해 만든 더미 함수.
    # time.sleep()으로 처리 시간을 고정해 성능 테스트(15초 기준)를 검증하는 데 사용됨.
    time.sleep(processing_time_sec)
    return np.zeros_like(data_array)  # 실제 세그멘테이션 대신 0 배열 반환


def generate_3d_viewer_performance(seg_map: np.ndarray, processing_time_sec: float = 4.0) -> bool:
    """3D 뷰어 렌더링 성능 시뮬레이션 (목표 5초 이내)"""
    # 3D 렌더링 엔진의 처리 시간을 테스트 환경에서 모방하기 위한 더미 함수.
    # 실제 렌더링 대신 sleep()으로 고정 시간을 소모하게 하여,
    # "5초 이하 렌더링"이라는 제품 성능 기준을 단위 테스트로 검증하기 위한 목적.
    time.sleep(processing_time_sec)
    return True


# ICV 계산 로직 단위 테스트
def test_icv_calculation_accuracy():
    """ICV 실시간 계산 값이 정확한지 검증"""
    dummy_data = np.ones((10, 10, 10)) # 총 볼륨 1000
    icv_result = calculate_icv(dummy_data)

    # ICV 계산 결과가 비정상(0 이하)로 나오지 않는지 확인하는 기본 단위 테스트.
    # 실제 서비스에서는 엔지니어가 제공한 공식과 비교하여 계산 정확도를 검증해야 함.
    # 예: assert icv_result == pytest.approx(expected_value)

# 성능 (속도) 목표 단위 테스트 (PRE_03)
@pytest.mark.performance
def test_segmentation_speed():
    """해마 세그멘테이션이 15초 이내에 완료되는지 검증"""
    data = np.random.rand(128, 128, 128)
    TARGET_TIME_SEC = 15.0 # 목표 시간

    # 세그멘테이션 처리 시간이 요구사항(15초 이내)을 만족하는지 확인하는 테스트.
    # 임상/서비스 환경에서 응답 속도가 너무 느려지지 않도록 성능 기준을 검증하는 목적.

    start_time = time.time()
    segment_hippocampus_performance(data) # 14초 시뮬레이션
    elapsed_time = time.time() - start_time

    assert elapsed_time < TARGET_TIME_SEC, f"세그멘테이션 속도 초과: {elapsed_time:.2f}초 > {TARGET_TIME_SEC}초"


@pytest.mark.performance
def test_3d_viewer_speed():
    """3D 뷰어 렌더링이 5초 이내에 완료되는지 검증"""
    seg_map = np.random.randint(0, 2, (64, 64, 64))
    TARGET_TIME_SEC = 5.0 # 목표 시간

    # 3D 뷰어의 렌더링 속도가 5초 기준을 넘지 않는지 확인하는 단위 테스트.
    # 실제 제품에서 사용자 경험(UX) 저하를 방지하기 위해 반드시 수행해야 하는 성능 검증 항목.

    start_time = time.time()
    generate_3d_viewer_performance(seg_map) # 4초 시뮬레이션
    elapsed_time = time.time() - start_time

    assert elapsed_time < TARGET_TIME_SEC, f"3D 뷰어 렌더링 속도 초과: {elapsed_time:.2f}초 > {TARGET_TIME_SEC}초"

# QA 코드 로직 검증
def test_compute_metrics_logic():
    """QA 코드의 AUC, 재현율 계산 로직이 정확한지 검증"""
    y_true = np.array([0, 0, 1, 1])
    y_score = np.array([0.1, 0.2, 0.9, 0.99])
    metrics = compute_metrics(y_true, y_score)

    # QA 코드에서 사용하는 성능 지표(AUC, Sensitivity 등)의 계산이 올바른지 확인하는 테스트.
    # 지표 계산 오류는 모델 평가 전체를 왜곡할 수 있으므로 반드시 검증 필요.
    # 아래는 AUC 계산이 정상적으로 동작하는지 확인하는 예시.

    assert metrics['AUC'] == pytest.approx(1.0), "AUC 계산 오류"
